<center><h1>Keyword Spotting (KWS): hey_dragon</h1></center>

Train a neural keyword spotter on Speech Commands using **log-mel spectrogram** inputs and a **CNN** (plus an optional Depthwise-Separable CNN).



## Setup checklist

### Environment
- [x] Python `3.11.5`
- [x] `torch>=1.8`
- [x] `torchaudio>=0.8` (match your `torch` build)
- [x] `librosa>=0.8`

**Install (example, CPU-only):**
- `python -m pip install --upgrade pip`
- `python -m pip install --index-url https://download.pytorch.org/whl/cpu "torch>=1.8" "torchaudio>=0.8"`
- `python -m pip install "librosa>=0.8" numpy matplotlib onnx onnxruntime sounddevice`

**Verify:**
- `python -c "import sys, torch, torchaudio, librosa; print(sys.version.split()[0]); print('torch', torch.__version__); print('torchaudio', torchaudio.__version__); print('librosa', librosa.__version__)"`

### Host machine
- Create + activate a venv, then install deps: `numpy`, `torch`, `onnx`, `onnxruntime`, `librosa`, `sounddevice`, `matplotlib`.
### UNOQ4 (target device)
- Confirm audio devices:
  - `arecord -l` and `aplay -l`
- Confirm speaker output works (adjust `-D` as needed):
  - `speaker-test -D plughw:CARD=Seri,DEV=0 -t sine -f 1000 -l 1 -p 20000`
- If using RPC (MCU heart):
  - Ensure router socket exists: `ls -la /var/run/arduino-router.sock`
  - Ensure Python has `msgpack` in your UNOQ4 venv: `python3 -c "import msgpack; print('msgpack_ok')"`
  - Ensure MCU exposes `flash_heart` (upload `rpc/MCU/sketch/rpc_hearts` if needed).

### Notes
- Run streaming inference from the Export section after you generate ONNX artifacts.


## Developer quickstart

1) Prepare the dataset folder:
   - Path: `projects/wake_word/speech_commands_hey_dragon/`
   - Include: your keyword folder (positives), a few negative word folders, and the split files `validation_list.txt` + `testing_list.txt`
2) Run the notebook top to bottom to train and evaluate.
3) Export ONNX + labels, then run streaming inference on UNOQ4 using `kws_keyword_beep_onnx.py`.


## Dataset: Speech Commands
This notebook expects the Speech Commands data to be available locally (downloaded + extracted ahead of time).

- Default dataset folder: `speech_commands_hey_dragon/`

Note: some repo copies may already include a prepared subset at `speech_commands_hey_dragon/`.


### Imports and reproducibility
**Goal**
- Import the core libraries used throughout the notebook (audio I/O, feature extraction, modeling, and evaluation).
- Set a deterministic seed for reproducible runs.



In [ ]:
import json, os, random
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import librosa
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from torch.utils.data import DataLoader, Dataset

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


### Dataset root (SPEECH_COMMANDS_DIR)
**Goal**
- Locate your extracted Speech Commands dataset.
- (Optional) create `speech_commands_hey_dragon/` from a full Speech Commands root using `make_min_dataset.py`.

**Notes**
- Downloading from the internet may be blocked in restricted environments; the repo includes a local archive.


In [ ]:
def _speech_commands_ok(root: Path) -> bool:
    return (root / 'validation_list.txt').exists() and (root / 'testing_list.txt').exists()

# For good false-positive behavior, your dataset root must include:
# - hey_dragon/ (positives)
# - a few negative word folders (to form _unknown_)
# - _background_noise_/ (optional but recommended)
# - validation_list.txt + testing_list.txt

SPEECH_COMMANDS_DIR = Path(os.environ.get('SPEECH_COMMANDS_DIR', 'speech_commands_hey_dragon'))

if not _speech_commands_ok(SPEECH_COMMANDS_DIR):
    raise FileNotFoundError(
        f"Expected Speech Commands dataset at: {SPEECH_COMMANDS_DIR.resolve()}\n"
        "Set SPEECH_COMMANDS_DIR to your dataset root folder, or to a minimal subset folder (recommended).\n"
        "Tip: build the minimal subset via: python3 make_min_dataset.py --src /path/to/speech_commands_root --dst speech_commands_hey_dragon\n"
        "Expected files: validation_list.txt, testing_list.txt, and label subfolders with .wav files."
    )

SPEECH_COMMANDS_DIR


## Split: train/val/test
Speech Commands ships with `validation_list.txt` and `testing_list.txt`. We use those lists and treat everything else as training.


### Official split lists
**Goal**
- Read the official split lists into fast lookup sets.


In [ ]:
def _read_list(path: Path) -> set[str]:
    lines = path.read_text(encoding='utf-8').splitlines()
    return {line.strip().replace(os.sep, '/') for line in lines if line.strip()}

val_list = _read_list(SPEECH_COMMANDS_DIR / 'validation_list.txt')
test_list = _read_list(SPEECH_COMMANDS_DIR / 'testing_list.txt')

# For any custom labels you add locally (e.g. hey_dragon), Speech Commands'
# official split lists won't include them. Create a deterministic 80/10/10
# split in-memory so training/metrics work without editing dataset files.
CUSTOM_LABEL_SPLIT_FRAC = (0.10, 0.10)  # (val, test)
custom_labels = ['hey_dragon']
existing = val_list | test_list
for label in custom_labels:
    wavs = sorted((SPEECH_COMMANDS_DIR / label).glob('*.wav'))
    rels = [p.relative_to(SPEECH_COMMANDS_DIR).as_posix() for p in wavs]
    if not rels:
        continue
    if any(r in existing for r in rels):
        continue
    rng = np.random.default_rng(SEED + (hash(label) % 10_000_000))
    idx = np.arange(len(rels))
    rng.shuffle(idx)
    n_val = max(1, int(round(len(rels) * CUSTOM_LABEL_SPLIT_FRAC[0])))
    n_test = max(1, int(round(len(rels) * CUSTOM_LABEL_SPLIT_FRAC[1])))
    val_idx = set(idx[:n_val])
    test_idx = set(idx[n_val:n_val + n_test])
    for i, rel in enumerate(rels):
        if i in val_idx:
            val_list.add(rel)
        elif i in test_idx:
            test_list.add(rel)

len(val_list), len(test_list)


## Log-mel Features
Neural KWS models usually take a time-frequency representation (log-mel, MFCC, etc.) as input.

Here we use:
- 1 second clips at 16 kHz
- log-mel spectrogram with 40 mel bins


### Audio and feature hyperparameters
**Goal**
- Define waveform and spectrogram settings.


In [ ]:
SAMPLE_RATE = 16_000
CLIP_SAMPLES = SAMPLE_RATE  # 1 second

N_FFT = 512
HOP_LENGTH = 160  # 10ms at 16kHz
N_MELS = 40
FMIN = 20
FMAX = SAMPLE_RATE // 2

# Numerical stability for log
LOG_EPS = 1e-6


### Fixed-length audio loader and log-mel extraction
**Goal**
- Load audio as a 1s mono clip.
- Compute a log-mel spectrogram tensor shaped `[1, n_mels, n_frames]`.


In [ ]:
def load_audio_1s(path: Path, *, sample_rate: int = SAMPLE_RATE, clip_samples: int = CLIP_SAMPLES, rng: np.random.Generator | None = None) -> np.ndarray:
    y, _ = librosa.load(path, sr=sample_rate, mono=True)
    y = y.astype(np.float32)

    if len(y) < clip_samples:
        y = np.pad(y, (0, clip_samples - len(y)))
        return y

    if len(y) > clip_samples:
        max_offset = len(y) - clip_samples
        if rng is None:
            offset = max_offset // 2
        else:
            offset = int(rng.integers(0, max_offset + 1))
        y = y[offset : offset + clip_samples]

    return y


def log_mel_spectrogram(y: np.ndarray, *, sample_rate: int = SAMPLE_RATE) -> torch.Tensor:
    # mel: [n_mels, n_frames]
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sample_rate,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0,
    ).astype(np.float32)

    log_mel = np.log(mel + LOG_EPS)

    # Normalize per-example for stability
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)

    # [1, n_mels, n_frames]
    return torch.from_numpy(log_mel).unsqueeze(0)


# Shape sanity check
x = log_mel_spectrogram(np.zeros(CLIP_SAMPLES, dtype=np.float32))
x.shape


## Dataset and DataLoaders
We mirror the baseline notebook’s label strategy:
- **Target words** (1 command: hey_dragon)
- **_unknown_** (all other words, balanced roughly to the number of training targets)
- **_silence_** (random 1s clips from `_background_noise_`)


### Target labels and balancing knobs
**Goal**
- Choose a 1-keyword task and optional unknown/silence classes.


In [ ]:
# Focused 3-class task:
#   _silence_, hey_dragon, _unknown_
TARGET_WORDS = ['hey_dragon']

INCLUDE_SILENCE = True
UNKNOWN_LABEL = '_unknown_'
SILENCE_LABEL = '_silence_'

MAX_PER_LABEL = 1000  # None for full dataset (slower)
TARGET_WORDS


### Discover labels in the dataset
**Goal**
- Enumerate label directories under the dataset root.


In [ ]:
def list_label_dirs(root: Path) -> list[str]:
    labels: list[str] = []
    for p in root.iterdir():
        if not p.is_dir():
            continue
        name = p.name
        if name.startswith('.'):
            continue
        if name in ['_background_noise_']:
            continue
        labels.append(name)
    return sorted(labels)

all_labels = list_label_dirs(SPEECH_COMMANDS_DIR)
neg_labels = [l for l in all_labels if l not in TARGET_WORDS]
if not neg_labels:
    print(
        "WARNING: no negative word folders found; _unknown_ will be empty. "
        "Add a few non-keyword word dirs (or use make_min_dataset.py)."
    )
missing = sorted(set(TARGET_WORDS) - set(all_labels))
if missing:
    raise ValueError(f"TARGET_WORDS not found under dataset root: {missing}")

len(all_labels), all_labels[:20]


### Build Example records and assign splits
**Goal**
- Build a list of `(path, label, split)` examples using official split files.


In [ ]:
RNG = np.random.default_rng(SEED)


def iter_wavs(label: str) -> Iterable[Path]:
    return (SPEECH_COMMANDS_DIR / label).glob('*.wav')


def relpath_under_root(path: Path) -> str:
    try:
        return path.relative_to(SPEECH_COMMANDS_DIR).as_posix()
    except ValueError:
        return path.as_posix()


def split_of(rel: str) -> str:
    if rel in val_list:
        return 'val'
    if rel in test_list:
        return 'test'
    return 'train'


@dataclass(frozen=True)
class Example:
    path: Path
    label: str
    split: str


def build_examples(*, rng: np.random.Generator = RNG) -> list[Example]:
    examples: list[Example] = []

    # Target words
    for label in TARGET_WORDS:
        files = sorted(iter_wavs(label))
        if MAX_PER_LABEL is not None and len(files) > MAX_PER_LABEL:
            idx = rng.choice(len(files), size=MAX_PER_LABEL, replace=False)
            files = [files[i] for i in idx]

        for p in files:
            rel = relpath_under_root(p)
            examples.append(Example(path=p, label=label, split=split_of(rel)))

    # Unknown = everything else (balanced roughly to the number of training targets)
    unknown_labels = [l for l in all_labels if l not in TARGET_WORDS]
    unknown_files: list[Path] = []
    for l in unknown_labels:
        unknown_files.extend(list(iter_wavs(l)))
    unknown_files = sorted(unknown_files)

    target_train = sum(1 for e in examples if e.split == 'train')
    unknown_keep = min(len(unknown_files), target_train)
    if unknown_keep > 0:
        idx = rng.choice(len(unknown_files), size=unknown_keep, replace=False)
        selected = [unknown_files[i] for i in idx]
        rng.shuffle(selected)

        # The official Speech Commands val/test lists may not include custom negative-word folders.
        # If we used split_of(rel) here, _unknown_ can end up train-only and be empty in val/test.
        if unknown_keep >= 3:
            n_val = max(1, int(round(unknown_keep * 0.1)))
            n_test = max(1, int(round(unknown_keep * 0.1)))
            n_train = unknown_keep - n_val - n_test
            if n_train < 1:
                n_train = 1
                if n_val > 1:
                    n_val -= 1
                elif n_test > 1:
                    n_test -= 1
        else:
            n_train, n_val, n_test = unknown_keep, 0, 0

        splits = [
            ('train', selected[:n_train]),
            ('val', selected[n_train : n_train + n_val]),
            ('test', selected[n_train + n_val :]),
        ]
        for split, items in splits:
            for p in items:
                examples.append(Example(path=p, label=UNKNOWN_LABEL, split=split))

    # Silence samples from background noise (added to all splits)
    if INCLUDE_SILENCE:
        noise_dir = SPEECH_COMMANDS_DIR / '_background_noise_'
        if noise_dir.exists():
            noise_files = sorted(noise_dir.glob('*.wav'))
            if noise_files:
                silence_total = min(max(600, len(TARGET_WORDS) * 150), target_train)
                n_train = int(round(silence_total * 0.8))
                n_val = int(round(silence_total * 0.1))
                n_test = silence_total - n_train - n_val

                for split, n in [('train', n_train), ('val', n_val), ('test', n_test)]:
                    for _ in range(n):
                        p = noise_files[int(rng.integers(0, len(noise_files))) ]
                        examples.append(Example(path=p, label=SILENCE_LABEL, split=split))

    return examples


examples = build_examples()

from collections import Counter
Counter((e.split, e.label) for e in examples)


### PyTorch Dataset
**Goal**
- Load a waveform, compute log-mel, and return `(features, class_index)`.

**Notes**
- For `_silence_`, we sample a random 1s window from a background-noise file.


In [ ]:
# Stable label ordering (important for metrics + export)
have_unknown = any(e.label == UNKNOWN_LABEL for e in examples)
if not have_unknown:
    print(
        "WARNING: _unknown_ has 0 examples. Metrics may show support=0 and the model will not learn negatives. "
        "Add a few non-keyword word folders (negatives) under the dataset root."
    )

label_order = TARGET_WORDS + ([UNKNOWN_LABEL] if have_unknown else [])
if INCLUDE_SILENCE:
    label_order = [SILENCE_LABEL] + label_order

label_to_index = {lab: i for i, lab in enumerate(label_order)}


def _random_clip_from_noise(path: Path, *, rng: np.random.Generator) -> np.ndarray:
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y = y.astype(np.float32)

    if len(y) <= CLIP_SAMPLES:
        return np.pad(y, (0, max(0, CLIP_SAMPLES - len(y))))

    max_offset = len(y) - CLIP_SAMPLES
    offset = int(rng.integers(0, max_offset + 1))
    return y[offset : offset + CLIP_SAMPLES]


class KWSDataset(Dataset):
    def __init__(self, examples: list[Example], *, rng_seed: int = SEED):
        self.examples = examples
        self.rng = np.random.default_rng(rng_seed)

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int):
        ex = self.examples[idx]
        if ex.label == SILENCE_LABEL:
            y = _random_clip_from_noise(ex.path, rng=self.rng)
        else:
            y = load_audio_1s(ex.path, rng=self.rng)

        x = log_mel_spectrogram(y)
        y_idx = label_to_index[ex.label]
        return x, y_idx


def select_split(split: str) -> list[Example]:
    return [e for e in examples if e.split == split]

train_ex = select_split('train')
val_ex = select_split('val')
test_ex = select_split('test')

len(train_ex), len(val_ex), len(test_ex), label_order


### DataLoaders
**Goal**
- Build DataLoaders for train/val/test.


In [ ]:
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64"))
NUM_WORKERS = 0  # increase if your environment supports it

train_ds = KWSDataset(train_ex, rng_seed=SEED)
val_ds = KWSDataset(val_ex, rng_seed=SEED + 1)
test_ds = KWSDataset(test_ex, rng_seed=SEED + 2)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))

# Inspect one batch
xb, yb = next(iter(train_loader))
xb.shape, yb.shape, xb.dtype


## Model Architecture (PyTorch)
We define two architectures you can swap between:

1. `KWSConvNet`: a simple Conv/BN/ReLU stack.
2. `KWSDscnn`: depthwise-separable conv blocks (common in small-footprint KWS).

Both consume an input shaped `[B, 1, n_mels, n_frames]`.


### Model definitions
**Goal**
- Implement compact CNN architectures for keyword classification.


In [ ]:
def count_params(model: nn.Module) -> int:
    """Count trainable parameters in a PyTorch module.

    Args:
        model: PyTorch module to inspect.

    Returns:
        Total number of parameters with ``requires_grad=True``.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class KWSConvNet(nn.Module):
    """Small baseline CNN for log-mel KWS inputs.

    Baseline CNN uses standard conv/BN/ReLU + pooling stack (no depthwise-separable blocks).
    Expects input shaped like ``(N, 1, n_mels, n_frames)``.
    """
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


class DepthwiseSeparableConv2d(nn.Module):
    """Depthwise-separable conv block (depthwise + pointwise + BN + ReLU).

    Used to build lightweight KWS models (DS-CNN style).
    """
    def __init__(self, in_ch: int, out_ch: int, *, kernel_size: int = 3, stride: int = 1, padding: int = 1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size=kernel_size, stride=stride, padding=padding, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return F.relu(x, inplace=True)


class KWSDscnn(nn.Module):
    """Lightweight DS-CNN-style model for KWS on log-mel features.

    DS-CNN-style uses depthwise-separable convolutions to cut params/compute vs standard convs.
    Expects input shaped like ``(N, 1, n_mels, n_frames)``.
    """
    def __init__(self, num_classes: int):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.blocks = nn.Sequential(
            DepthwiseSeparableConv2d(64, 64),
            nn.MaxPool2d(kernel_size=2),
            DepthwiseSeparableConv2d(64, 128),
            nn.MaxPool2d(kernel_size=2),
            DepthwiseSeparableConv2d(128, 128),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.blocks(x)
        return self.head(x)


NUM_CLASSES = len(label_order)
model = KWSDscnn(NUM_CLASSES).to(DEVICE)  # swap to KWSConvNet(NUM_CLASSES) if you prefer

print(model)
print('trainable params:', count_params(model))


## Train


### Training utilities
**Goal**
- Implement `train_one_epoch` and `evaluate` loops.


In [ ]:
def accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        logits = model(xb)
        loss = F.cross_entropy(logits, yb)

        total_loss += loss.item() * yb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return {
        'loss': total_loss / max(1, total),
        'acc': total_correct / max(1, total),
    }


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> dict:
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return {
        'loss': total_loss / max(1, total),
        'acc': total_correct / max(1, total),
    }


### Run training
**Goal**
- Train for a few epochs and track validation accuracy.

**Tips**
- Start with a small `MAX_PER_LABEL` while iterating.
- If you have a GPU, training is much faster.


In [ ]:
import time

# Note: re-running this cell continues training from the current model weights.
# For a fresh run from scratch, restart the kernel OR set RESET_MODEL=1 and re-run.
RESET_MODEL = int(os.getenv("RESET_MODEL", "0"))

def reset_model_weights(m: nn.Module) -> None:
    """Reset parameters of all submodules that implement ``reset_parameters()``."""
    def _reset(sub: nn.Module) -> None:
        if hasattr(sub, "reset_parameters"):
            sub.reset_parameters()
    m.apply(_reset)

if RESET_MODEL:
    reset_model_weights(model)
    print("Reset model weights")

EPOCHS = int(os.getenv("EPOCHS", "5"))
LR = float(os.getenv("LR", "1e-3"))
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []


def fmt4(x):
    if isinstance(x, (float, np.floating)):
        return float(f"{float(x):.4f}")
    return x


for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    tr = train_one_epoch(model, train_loader, optimizer)
    va = evaluate(model, val_loader)
    epoch_s = time.perf_counter() - t0

    row = {
        "epoch": epoch,
        "epoch_s": epoch_s,
        **{f"train_{k}": v for k, v in tr.items()},
        **{f"val_{k}": v for k, v in va.items()},
    }
    row = {k: fmt4(v) for k, v in row.items()}
    history.append(row)
    print(row)

history[-1]


## Evaluate


### Test-set report + confusion matrix
**Goal**
- Compute a classification report and confusion matrix on the held-out test split.


In [ ]:
@torch.no_grad()
def predict_all(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    y_true = []
    y_pred = []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        y_pred.append(preds)
        y_true.append(yb.numpy())

    return np.concatenate(y_true), np.concatenate(y_pred)


y_true, y_pred = predict_all(model, test_loader)

all_class_ids = list(range(len(label_order)))
print(classification_report(y_true, y_pred, labels=all_class_ids, target_names=label_order, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_class_ids)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm, display_labels=label_order).plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
ax.set_title('Keyword Spotting Confusion Matrix (PyTorch)')
plt.show()


## Export
We export:
- `pytorch_kws_model.pt`: `state_dict` + small metadata
- `pytorch_kws_labels.json`: label ordering used for training


### Save model and labels
**Goal**
- Save the trained model weights and label list for later inference.


In [ ]:
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / 'pytorch_kws_model.pt'
LABELS_PATH = OUTPUT_DIR / 'pytorch_kws_labels.json'

payload = {
    'model_type': model.__class__.__name__,
    'state_dict': model.state_dict(),
    'sample_rate': SAMPLE_RATE,
    'clip_samples': CLIP_SAMPLES,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'n_mels': N_MELS,
    'fmin': FMIN,
    'fmax': FMAX,
}

torch.save(payload, MODEL_PATH)
LABELS_PATH.write_text(json.dumps(label_order, indent=2), encoding='utf-8')

MODEL_PATH, LABELS_PATH


## Inference Example
Run the exported model on a single `.wav` file.


### Load + predict
**Goal**
- Load the saved weights.
- Run a forward pass on one file.


In [ ]:
# Pick any WAV under the dataset (or set your own path)
example_wav = next((SPEECH_COMMANDS_DIR / TARGET_WORDS[0]).glob('*.wav'))
example_wav


In [ ]:
labels = json.loads(LABELS_PATH.read_text(encoding='utf-8'))
loaded = torch.load(MODEL_PATH, map_location=DEVICE)

# Recreate the model (match MODEL_TYPE)
if loaded['model_type'] == 'KWSConvNet':
    infer_model = KWSConvNet(len(labels))
elif loaded['model_type'] == 'KWSDscnn':
    infer_model = KWSDscnn(len(labels))
else:
    raise ValueError(f"Unknown model_type: {loaded['model_type']}")

infer_model.load_state_dict(loaded['state_dict'])
infer_model.to(DEVICE)
infer_model.eval()

# Feature + predict
rng = np.random.default_rng(SEED)
y = load_audio_1s(example_wav, rng=rng)
x = log_mel_spectrogram(y).unsqueeze(0).to(DEVICE)  # [1, 1, n_mels, n_frames]

with torch.no_grad():
    logits = infer_model(x)
    probs = logits.softmax(dim=1).squeeze(0).cpu().numpy()

pred_idx = int(probs.argmax())
print('file:', example_wav)
print('pred:', labels[pred_idx])
print('top5:')
for i in probs.argsort()[-5:][::-1]:
    print(f"  {labels[int(i)]:>10s}: {probs[int(i)]:.3f}")


### Export the saved PyTorch checkpoint to ONNX

Option A (direct): use PyTorch's ONNX exporter (`torch.onnx.export`).
Option B (recommended for this repo): run `export_pytorch_to_onnx.py` to also emit the config JSON used by the streaming demo.




In [ ]:
# Option A: direct export with torch.onnx.export (no helper script)
# Note: this writes the ONNX file, but does NOT write the config JSON used by kws_keyword_beep_onnx.py.
# Export uses the legacy exporter (dynamo=False) for compatibility and avoids dynamic axes (fixed 1s input).

loaded = torch.load(MODEL_PATH, map_location="cpu")
labels = json.loads(LABELS_PATH.read_text(encoding="utf-8"))

# Recreate the model (match the checkpoint)
if loaded["model_type"] == "KWSConvNet":
    export_model = KWSConvNet(len(labels))
elif loaded["model_type"] == "KWSDscnn":
    export_model = KWSDscnn(len(labels))
else:
    raise ValueError(f"Unknown model_type: {loaded['model_type']}")

export_model.load_state_dict(loaded["state_dict"])
export_model.eval()

onnx_path = OUTPUT_DIR / "pytorch_kws_model.onnx"

# Dummy input: (N, 1, n_mels, n_frames)
n_frames = int(log_mel_spectrogram(np.zeros(CLIP_SAMPLES, dtype=np.float32)).shape[-1])
dummy_x = torch.zeros(1, 1, N_MELS, n_frames, dtype=torch.float32)

torch.onnx.export(
    export_model,
    dummy_x,
    str(onnx_path),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["x"],
    output_names=["logits"],
    dynamo=False,
)

onnx_path


In [ ]:
import subprocess
import sys

onnx_path = OUTPUT_DIR / 'pytorch_kws_model.onnx'
config_path = OUTPUT_DIR / 'pytorch_kws_config.json'

cmd = [
    sys.executable,
    'export_pytorch_to_onnx.py',
    '--pt',
    str(MODEL_PATH),
    '--labels',
    str(LABELS_PATH),
    '--onnx',
    str(onnx_path),
    '--config',
    str(config_path),
]

subprocess.check_call(cmd)
onnx_path, config_path


## Qualcomm AI Hub Workbench (optional)

Use Qualcomm® AI Hub Workbench to profile (and optionally compile) the exported ONNX model.

**Security**: do not paste API tokens into this notebook. Use a password manager and pass tokens via environment variables (recommended) or configure the CLI on your machine.


In [ ]:
# AI Hub Workbench quickstart (safe by default: does not run unless ENABLE_QAI_HUB=1)
import os
from pathlib import Path

import numpy as np

ENABLE_QAI_HUB = int(os.getenv("ENABLE_QAI_HUB", "0"))
if not ENABLE_QAI_HUB:
    print("AI Hub section is disabled by default.")
    print("To enable (recommended):")
    print("  1) qai-hub configure --api_token <token from password manager>")
    print("  2) export ENABLE_QAI_HUB=1")
    print("  3) start Jupyter from that same shell")
    print("Ephemeral option: export QAI_HUB_API_TOKEN=<token> before starting Jupyter.")
else:
    import qai_hub as hub

    token = os.getenv("QAI_HUB_API_TOKEN")
    client = hub.Client(hub.ClientConfig(api_token=token)) if token else hub.Client()

    onnx_file = Path(onnx_path)
    if not onnx_file.exists():
        raise FileNotFoundError(
            f"Missing ONNX file: {onnx_file}. Run the ONNX export cells first."
        )

    device_name = os.getenv("QAI_HUB_DEVICE", "").strip()
    if not device_name:
        print("Set QAI_HUB_DEVICE to an exact device name (see: `qai-hub list-devices`).")
        raise RuntimeError("Missing QAI_HUB_DEVICE")
    device = hub.Device(device_name)

    # Input spec for compilation (match the ONNX model's input name + shape)
    input_name = (os.getenv("QAI_HUB_INPUT_NAME", "x").strip() or "x")
    n_frames = int(log_mel_spectrogram(np.zeros(CLIP_SAMPLES, dtype=np.float32)).shape[-1])
    input_shape = (1, 1, int(N_MELS), n_frames)
    input_specs = {input_name: input_shape}

    name_base = os.getenv("QAI_HUB_JOB_NAME", f"kws_{onnx_file.stem}")
    do_compile = int(os.getenv("QAI_HUB_COMPILE", "1"))
    compile_options = os.getenv("QAI_HUB_COMPILE_OPTIONS", "--target_runtime tflite")
    profile_options = os.getenv("QAI_HUB_PROFILE_OPTIONS", "")
    wait = int(os.getenv("QAI_HUB_WAIT", "1"))

    model_for_profile = str(onnx_file)
    target_model = None

    if do_compile:
        compile_job = client.submit_compile_job(
            model=str(onnx_file),
            device=device,
            name=f"{name_base}_compile",
            input_specs=input_specs,
            options=compile_options,
        )
        print("Submitted compile job:", compile_job.url)
        if wait:
            compile_job.wait()
        target_model = compile_job.get_target_model()
        model_for_profile = target_model
    else:
        print("Skipping compile; profiling the source model as-is.")

    profile_job = client.submit_profile_job(
        model=model_for_profile,
        device=device,
        name=f"{name_base}_profile",
        options=profile_options,
    )
    print("Submitted profile job:", profile_job.url)
    if wait:
        profile_job.wait()

    out_dir = Path(OUTPUT_DIR) / "qai_hub"

    if int(os.getenv("QAI_HUB_DOWNLOAD", "0")):
        out_dir.mkdir(parents=True, exist_ok=True)
        profile_job.download_results(str(out_dir))
        print("Downloaded profile results to:", out_dir)

    if target_model is not None and int(os.getenv("QAI_HUB_DOWNLOAD_COMPILED_MODEL", "0")):
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{name_base}_compiled"
        filename = target_model.download(str(out_path))
        print("Downloaded compiled model to:", filename)


## Streaming inference demo (ONNX Runtime)

Use `kws_keyword_beep_onnx.py` to run streaming inference from microphone input. This script is intended to be run on the target device (UNOQ4) where ALSA audio devices are available.

Tip: start by running with `--print-keyword-prob --show-unknown-top` for a 60s silent baseline, then set `--threshold` based on the observed distribution.

Tip: `RMS` is the root-mean-square amplitude of the current audio ring buffer (the last ~1 second of mic audio after any resampling). It's computed as `rms = sqrt(mean(ring**2))` where `ring` is float32 waveform samples in roughly `[-1.0, +1.0]`.
- Lower RMS = quieter / closer to silence (mostly noise floor)
- Higher RMS = louder audio (speech, bumps, fan noise)


In [ ]:
# Run streaming inference demo (kws_keyword_beep_onnx.py)
#
# This cell prints example commands; it does not start the infinite loop unless you set RUN_STREAMING_DEMO=1.
import os
import shlex
import subprocess
import sys

cmd_help = [sys.executable, 'kws_keyword_beep_onnx.py', '--help']
cmd_list = [sys.executable, 'kws_keyword_beep_onnx.py', '--output-dir', str(OUTPUT_DIR), '--list-labels']

print('help:', shlex.join(cmd_help))
print('labels:', shlex.join(cmd_list))

cmd_run = [
    sys.executable,
    'kws_keyword_beep_onnx.py',
    '--output-dir',
    str(OUTPUT_DIR),
    '--keyword',
    'hey_dragon',
    '--threshold',
    '0.90',
    '--stable-n',
    '5',
    '--cooldown-steps',
    '20',
    '--print-keyword-prob',
    '--show-unknown-top',
    '--print-any-detect',
]
print('run:', shlex.join(cmd_run))

if os.getenv('RUN_STREAMING_DEMO', '').strip() == '1':
    subprocess.check_call(cmd_run)
